# 🛡️ SMS Spam Classifier — Interactive GUI
**Egyptian Chinese University — Data Mining Assignment #1**

**Models:** Naive Bayes | Logistic Regression | KNN  
**Feature Engineering:** TF-IDF · **Optimization:** GridSearchCV + 5-Fold CV

> ▶️ Run all cells with **Ctrl+Shift+P → "Run All"** — the GUI will appear at the bottom


In [3]:
import pandas as pd
import numpy as np
import re, string, warnings, zipfile, io, os
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("✅ All libraries imported!")


✅ All libraries imported!


In [4]:
STOP_WORDS = set([
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'yourself','yourselves','he','him','his','himself','she','her','hers',
    'herself','it','its','itself','they','them','their','theirs','themselves',
    'what','which','who','whom','this','that','these','those','am','is','are',
    'was','were','be','been','being','have','has','had','having','do','does',
    'did','doing','a','an','the','and','but','if','or','because','as','until',
    'while','of','at','by','for','with','about','against','between','into',
    'through','during','before','after','above','below','to','from','up','down',
    'in','out','on','off','over','under','again','further','then','once','here',
    'there','when','where','why','how','all','both','each','few','more','most',
    'other','some','such','no','nor','not','only','own','same','so','than',
    'too','very','s','t','can','will','just','don','should','now','d','ll',
    'm','o','re','ve','y','ain','aren','couldn','didn','doesn','hadn','hasn',
    'haven','isn','ma','mightn','mustn','needn','shan','shouldn','wasn',
    'weren','won','wouldn'
])

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join([w for w in text.split() if w not in STOP_WORDS])

def load_spam_data():
    search_dirs = [os.getcwd(), os.path.expanduser('~/Downloads'), os.path.expanduser('~/Desktop')]
    for folder in search_dirs:
        csv_path = os.path.join(folder, 'spam.csv')
        if os.path.exists(csv_path):
            return pd.read_csv(csv_path, encoding='latin-1')
        for zip_name in ['archive.zip', 'sms-spam-collection-dataset.zip']:
            zip_path = os.path.join(folder, zip_name)
            if os.path.exists(zip_path):
                with zipfile.ZipFile(zip_path) as z:
                    with z.open('spam.csv') as zf:
                        return pd.read_csv(zf, encoding='latin-1')
    raise FileNotFoundError("❌ spam.csv not found! Place it in the same folder as this notebook.\nDownload: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset")

print("✅ Helper functions ready!")


✅ Helper functions ready!


In [5]:
print("📂 Loading dataset...")
df = load_spam_data()
df = df[['v1','v2']].copy()
df.columns = ['label','message']
df = df.drop_duplicates().reset_index(drop=True)
df['msg_length']      = df['message'].apply(len)
df['cleaned_message'] = df['message'].apply(clean_text)
df['label_encoded']   = df['label'].map({'ham': 0, 'spam': 1})
print(f"✅ Loaded {len(df):,} messages  |  Spam: {(df.label=='spam').sum()}  |  Ham: {(df.label=='ham').sum()}")

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_message'], df['label_encoded'],
    test_size=0.2, random_state=42, stratify=df['label_encoded']
)

print("🔠 Vectorizing with TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("🤖 Training Naive Bayes...")
nb_grid = GridSearchCV(MultinomialNB(), {'alpha':[0.01,0.1,0.5,1.0,2.0]}, cv=cv, scoring='f1', n_jobs=-1)
nb_grid.fit(X_train_tfidf, y_train)

print("🤖 Training Logistic Regression...")
lr_grid = GridSearchCV(LogisticRegression(random_state=42),
    {'C':[0.01,0.1,1,10,100],'solver':['lbfgs','liblinear'],'max_iter':[1000]},
    cv=cv, scoring='f1', n_jobs=-1)
lr_grid.fit(X_train_tfidf, y_train)

print("🤖 Training KNN...")
knn_grid = GridSearchCV(KNeighborsClassifier(),
    {'n_neighbors':[3,5,7,11],'metric':['euclidean','cosine'],'weights':['uniform','distance']},
    cv=cv, scoring='f1', n_jobs=-1)
knn_grid.fit(X_train_tfidf, y_train)

trained_models = {
    'Naive Bayes':         nb_grid.best_estimator_,
    'Logistic Regression': lr_grid.best_estimator_,
    'KNN':                 knn_grid.best_estimator_,
}

results = {}
for name, model in trained_models.items():
    yp = model.predict(X_test_tfidf)
    results[name] = {
        'accuracy':  accuracy_score(y_test, yp),
        'precision': precision_score(y_test, yp),
        'f1':        f1_score(y_test, yp),
        'y_pred':    yp,
        'cm':        confusion_matrix(y_test, yp),
    }

best_name = max(results, key=lambda k: results[k]['f1'])
print(f"\n🏆 Best model: {best_name}  (F1 = {results[best_name]['f1']*100:.2f}%)")
print("\n✅ All models trained! Scroll down to launch the GUI ⬇️")


📂 Loading dataset...
✅ Loaded 5,169 messages  |  Spam: 653  |  Ham: 4516
🔠 Vectorizing with TF-IDF...
🤖 Training Naive Bayes...
🤖 Training Logistic Regression...
🤖 Training KNN...

🏆 Best model: Logistic Regression  (F1 = 90.98%)

✅ All models trained! Scroll down to launch the GUI ⬇️


In [6]:
# ═══════════════════════════════════════════════
#        FULL INTERACTIVE GUI  —  v4 (Redesign)
# ═══════════════════════════════════════════════

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Fonts ──────────────────────────────────────
display(HTML('''<link href="https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Sans:wght@300;400;500&family=DM+Mono:wght@400;500&display=swap" rel="stylesheet">'''))

# ── CSS ────────────────────────────────────────
display(HTML('''
<style>
:root{--bg:#0a0b0f;--surf:#111318;--surf2:#181b22;--brd:rgba(255,255,255,.07);--brd2:rgba(255,255,255,.13);--tx:#e8eaf2;--mut:#6b7280;--acc:#5b8fff;--acc2:#4ade80;--dng:#ff5c5c;--r:12px;}
.sc-root{font-family:"DM Sans",sans-serif;background:var(--bg);color:var(--tx);border-radius:20px;overflow:hidden;}
.sc-hero{padding:26px 26px 18px;border-bottom:1px solid var(--brd);background:linear-gradient(140deg,#0e1220 0%,#111827 100%);position:relative;overflow:hidden;}
.sc-hero::before{content:"";position:absolute;top:-60px;right:-50px;width:200px;height:200px;border-radius:50%;background:radial-gradient(circle,rgba(91,143,255,.20) 0%,transparent 70%);pointer-events:none;}
.sc-hero::after{content:"";position:absolute;bottom:-40px;left:35%;width:150px;height:150px;border-radius:50%;background:radial-gradient(circle,rgba(74,222,128,.12) 0%,transparent 70%);pointer-events:none;}
.sc-title{font-family:"Syne",sans-serif;font-size:24px;font-weight:800;letter-spacing:-.5px;color:#fff;margin-bottom:4px;}
.sc-sub{font-size:11px;color:var(--mut);margin-bottom:13px;letter-spacing:.03em;}
.sc-tags{display:flex;flex-wrap:wrap;gap:5px;}
.sc-tag{font-size:10px;font-family:"DM Mono",monospace;padding:3px 9px;border-radius:99px;border:1px solid;letter-spacing:.04em;}
.t-blue{background:rgba(91,143,255,.12);border-color:rgba(91,143,255,.35);color:#7ba7ff;}
.t-green{background:rgba(74,222,128,.10);border-color:rgba(74,222,128,.3);color:#4ade80;}
.t-purple{background:rgba(168,139,250,.10);border-color:rgba(168,139,250,.3);color:#b48cf5;}
.sc-stats{display:grid;grid-template-columns:repeat(5,1fr);border-bottom:1px solid var(--brd);}
.sc-stat{padding:14px 10px;text-align:center;border-right:1px solid var(--brd);}
.sc-stat:last-child{border-right:none;}
.sv{font-family:"Syne",sans-serif;font-size:18px;font-weight:700;color:#fff;line-height:1;}
.sl{font-size:9px;color:var(--mut);margin-top:3px;letter-spacing:.04em;}
.sec-hd{font-family:"DM Mono",monospace;color:var(--acc);font-size:12px;border-bottom:1px solid var(--brd2);padding-bottom:6px;margin:0 0 12px;}
.rcard{border-radius:16px;padding:18px 20px;margin-top:12px;}
.rspam{background:linear-gradient(135deg,rgba(255,92,92,.13),rgba(180,30,30,.07));border:1px solid rgba(255,92,92,.38);}
.rham{background:linear-gradient(135deg,rgba(74,222,128,.13),rgba(22,163,74,.07));border:1px solid rgba(74,222,128,.38);}
.rtop{display:flex;align-items:center;gap:14px;}
.rdot{width:42px;height:42px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:18px;flex-shrink:0;}
.dspam{background:rgba(255,92,92,.2);}.dham{background:rgba(74,222,128,.2);}
.verd{font-family:"Syne",sans-serif;font-size:20px;font-weight:800;}
.vspam{color:#ff7070;}.vham{color:#4ade80;}
.vsub{font-size:11px;color:var(--mut);margin-top:3px;}
.cwrap{margin-top:12px;}
.chdr{display:flex;justify-content:space-between;font-size:10px;color:var(--mut);margin-bottom:4px;}
.ctrack{height:4px;background:rgba(255,255,255,.08);border-radius:99px;overflow:hidden;}
.cfill{height:100%;border-radius:99px;}
.fspam{background:linear-gradient(90deg,#ff5c5c,#ff9090);}.fham{background:linear-gradient(90deg,#22c55e,#4ade80);}
.tline{font-size:11px;color:var(--mut);font-family:"DM Mono",monospace;margin-top:9px;padding:7px 10px;background:rgba(0,0,0,.25);border-radius:7px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;}
.btable{width:100%;border-collapse:collapse;font-size:12px;}
.btable th{text-align:left;font-size:10px;letter-spacing:.06em;color:var(--mut);font-weight:500;padding:7px 11px;border-bottom:1px solid var(--brd);text-transform:uppercase;}
.btable td{padding:9px 11px;border-bottom:1px solid var(--brd);vertical-align:middle;color:var(--tx);}
.btable tr:last-child td{border-bottom:none;}
.pill{display:inline-block;font-size:10px;font-family:"DM Mono",monospace;padding:2px 8px;border-radius:99px;font-weight:500;letter-spacing:.03em;}
.pspam{background:rgba(255,92,92,.18);color:#ff7a7a;border:1px solid rgba(255,92,92,.35);}
.pham{background:rgba(74,222,128,.15);color:#4ade80;border:1px solid rgba(74,222,128,.3);}
.pgrid{display:grid;grid-template-columns:repeat(3,1fr);gap:9px;margin-bottom:14px;}
.pcard{background:var(--surf);border:1px solid var(--brd2);border-radius:13px;padding:14px 13px;}
.pcard.win{border-color:rgba(91,143,255,.4);background:rgba(91,143,255,.07);}
.ph{font-size:11px;font-weight:500;color:var(--tx);margin-bottom:10px;display:flex;align-items:center;gap:5px;}
.wb{font-size:9px;background:rgba(91,143,255,.2);color:var(--acc);padding:2px 7px;border-radius:99px;font-family:"DM Mono",monospace;border:1px solid rgba(91,143,255,.3);}
.mr{display:flex;justify-content:space-between;align-items:center;padding:4px 0;}
.mr+.mr{border-top:1px solid var(--brd);}
.mn{font-size:10px;color:var(--mut);}
.mv{font-size:12px;font-family:"DM Mono",monospace;color:var(--tx);font-weight:500;}
.mv.hi{color:var(--acc2);}
.cbar-wrap{background:var(--surf);border:1px solid var(--brd2);border-radius:13px;padding:16px;margin-bottom:12px;}
.bgroup{margin-bottom:13px;}
.bl{display:flex;justify-content:space-between;font-size:11px;margin-bottom:4px;}
.bn{color:var(--mut);font-size:10px;}
.bv{font-family:"DM Mono",monospace;color:var(--tx);font-size:11px;}
.btrack{height:5px;background:rgba(255,255,255,.06);border-radius:99px;overflow:hidden;}
.bfill{height:100%;border-radius:99px;}
.cmw{background:var(--surf);border:1px solid var(--brd2);border-radius:13px;padding:16px;}
.cmt{border-collapse:collapse;font-size:12px;}
.cmt th{color:var(--mut);padding:5px 13px;font-weight:400;font-size:10px;}
.cmt td{padding:9px 13px;text-align:center;font-family:"DM Mono",monospace;font-size:14px;}
.cok{color:var(--acc2);font-weight:600;}.cerr{color:var(--dng);}
.sc-footer{padding:11px 22px;border-top:1px solid var(--brd);background:var(--surf);font-size:10px;color:var(--mut);text-align:center;letter-spacing:.04em;}
/* Widget overrides */
.widget-textarea textarea,.widget-text input{background:var(--surf)!important;color:var(--tx)!important;border:1px solid var(--brd2)!important;border-radius:10px!important;font-family:"DM Sans",sans-serif!important;font-size:13px!important;}
.widget-dropdown select{background:var(--surf2)!important;color:var(--tx)!important;border:1px solid var(--brd2)!important;border-radius:8px!important;font-family:"DM Mono",monospace!important;font-size:12px!important;}
.widget-button button{background:var(--acc)!important;color:#fff!important;border:none!important;border-radius:10px!important;font-family:"DM Sans",sans-serif!important;font-size:13px!important;font-weight:500!important;padding:8px 20px!important;cursor:pointer!important;}
.btn-ghost button{background:transparent!important;color:var(--mut)!important;border:1px solid var(--brd2)!important;border-radius:8px!important;font-size:12px!important;}
.btn-ghost button:hover{color:var(--tx)!important;}
.sample-chip button{background:var(--surf2)!important;color:var(--mut)!important;border:1px solid var(--brd2)!important;border-radius:99px!important;font-size:11px!important;padding:4px 11px!important;}
.widget-label label{color:var(--mut)!important;font-size:10px!important;text-transform:uppercase!important;letter-spacing:.08em!important;font-weight:500!important;}
</style>
'''))

# ── Header HTML ────────────────────────────────
spam_count = (df['label']=='spam').sum()
ham_count  = (df['label']=='ham').sum()

display(HTML(f'''
<div class="sc-root">
  <div class="sc-hero">
    <div class="sc-title">&#x1F6E1; SMS Spam Classifier</div>
    <div class="sc-sub">TF-IDF &middot; GridSearchCV &middot; 5-Fold CV &middot; Egyptian Chinese University &mdash; Data Mining #1</div>
    <div class="sc-tags">
      <span class="sc-tag t-blue">Naive Bayes</span>
      <span class="sc-tag t-green">Logistic Regression</span>
      <span class="sc-tag t-purple">KNN</span>
      <span class="sc-tag t-blue">TF-IDF bigrams</span>
      <span class="sc-tag t-green">GridSearchCV</span>
      <span class="sc-tag t-purple">5-Fold CV</span>
    </div>
  </div>
  <div class="sc-stats">
    <div class="sc-stat"><div class="sv">{len(df):,}</div><div class="sl">Messages</div></div>
    <div class="sc-stat"><div class="sv">{spam_count/len(df)*100:.1f}%</div><div class="sl">Spam rate</div></div>
    <div class="sc-stat"><div class="sv">{results[best_name]['f1']*100:.1f}%</div><div class="sl">Best F1</div></div>
    <div class="sc-stat"><div class="sv">{results[best_name]['accuracy']*100:.1f}%</div><div class="sl">Best accuracy</div></div>
    <div class="sc-stat"><div class="sv">{best_name.split()[0]}</div><div class="sl">&#x1F3C6; Best model</div></div>
  </div>
</div>
'''))

# ═══════════════════ SECTION 1 — CLASSIFY ════════════════════
display(HTML("<div class='sec-hd' style='margin-top:16px;'>// &#x1F50D; Classify a Message</div>"))

model_dd = widgets.Dropdown(
    options=['Naive Bayes', 'Logistic Regression', 'KNN'],
    value=best_name,
    description='Model:',
    style={'description_width': '55px'},
    layout=widgets.Layout(width='260px')
)

msg_box = widgets.Textarea(
    placeholder='Type or paste any SMS message here…',
    layout=widgets.Layout(width='100%', height='110px')
)

classify_btn = widgets.Button(
    description='→ Classify message',
    layout=widgets.Layout(width='180px', height='38px')
)

clear_btn = widgets.Button(
    description='Clear',
    layout=widgets.Layout(width='80px', height='38px')
)
clear_btn.add_class('btn-ghost')

SAMPLES = [
    "Congratulations! You've won a FREE iPhone! Click here NOW to claim your prize! Text CLAIM to 87064.",
    "Hey, are you coming to the study group tonight? Let me know if you need the notes.",
    "Win £1000 cash prize now! Urgent: your account selected. Call FREE 0800 to claim. Expires midnight!"
]
s1_btn = widgets.Button(description='🔴 Spam 1', layout=widgets.Layout(width='100px'))
s2_btn = widgets.Button(description='🟢 Ham',    layout=widgets.Layout(width='90px'))
s3_btn = widgets.Button(description='🔴 Spam 2', layout=widgets.Layout(width='100px'))
for b in [s1_btn, s2_btn, s3_btn]:
    b.add_class('sample-chip')

result_out = widgets.Output()

def fill_sample(idx):
    def _f(b): msg_box.value = SAMPLES[idx]
    return _f
s1_btn.on_click(fill_sample(0))
s2_btn.on_click(fill_sample(1))
s3_btn.on_click(fill_sample(2))

def on_clear(b):
    with result_out:
        clear_output()
clear_btn.on_click(on_clear)

def on_classify(b):
    with result_out:
        clear_output(wait=True)
        msg = msg_box.value.strip()
        if not msg:
            display(HTML("<div style='color:#ff5c5c;font-size:13px;padding:8px 0;'>&#x26A0; Please enter a message first.</div>"))
            return
        model   = trained_models[model_dd.value]
        cleaned = clean_text(msg)
        vec     = tfidf.transform([cleaned])
        pred    = model.predict(vec)[0]
        try:
            proba = model.predict_proba(vec)[0]
            pct   = round(proba[pred] * 100, 1)
            conf_str = f"{pct}%"
            bar_w    = int(proba[pred] * 100)
        except:
            pct = ""
            conf_str = "N/A"
            bar_w    = 0
        tok = ' '.join(cleaned.split()[:15]) + ('…' if len(cleaned.split()) > 15 else '')
        if pred == 1:
            display(HTML(f"""
            <div class="rcard rspam">
              <div class="rtop">
                <div class="rdot dspam">&#x26A0;</div>
                <div>
                  <div class="verd vspam">SPAM</div>
                  <div class="vsub">Likely unsolicited &mdash; {model_dd.value}</div>
                </div>
              </div>
              <div class="cwrap">
                <div class="chdr"><span>Confidence</span><span>{conf_str}</span></div>
                <div class="ctrack"><div class="cfill fspam" style="width:{bar_w}%"></div></div>
              </div>
              <div class="tline">Tokens: {tok}</div>
            </div>"""))
        else:
            display(HTML(f"""
            <div class="rcard rham">
              <div class="rtop">
                <div class="rdot dham">&#x2713;</div>
                <div>
                  <div class="verd vham">HAM</div>
                  <div class="vsub">Looks like a normal message &mdash; {model_dd.value}</div>
                </div>
              </div>
              <div class="cwrap">
                <div class="chdr"><span>Confidence</span><span>{conf_str}</span></div>
                <div class="ctrack"><div class="cfill fham" style="width:{bar_w}%"></div></div>
              </div>
              <div class="tline">Tokens: {tok}</div>
            </div>"""))

classify_btn.on_click(on_classify)

display(widgets.VBox([
    model_dd,
    msg_box,
    widgets.HBox([classify_btn, clear_btn]),
    widgets.HBox([s1_btn, s2_btn, s3_btn]),
    result_out
], layout=widgets.Layout(gap='8px')))

# ═══════════════════ SECTION 2 — BATCH ════════════════════
batch_box = widgets.Textarea(
    placeholder="""Enter multiple messages, one per line:
Hey are you coming tonight?
WIN FREE cash now! Text CLAIM to 87064
See you at the meeting at 3pm""",
    layout=widgets.Layout(width='100%', height='100px')
)
batch_btn = widgets.Button(
    description='⚡ Classify all',
    layout=widgets.Layout(width='150px', height='38px')
)
batch_out = widgets.Output()

def on_batch(b):
    with batch_out:
        clear_output(wait=True)
        lines = [l.strip() for l in batch_box.value.strip().splitlines() if l.strip()]
        if not lines:
            display(HTML("<div style='color:#ff5c5c;font-size:13px;padding:8px 0;'>&#x26A0; Please enter at least one message.</div>"))
            return
        model   = trained_models[model_dd.value]
        cleaned = [clean_text(l) for l in lines]
        vecs    = tfidf.transform(cleaned)
        preds   = model.predict(vecs)
        try:
            probas = model.predict_proba(vecs)
            confs  = [f"{round(probas[i][p]*100,1)}%" for i,p in enumerate(preds)]
        except:
            confs  = ["N/A"] * len(preds)
        spam_n = sum(preds); ham_n = len(preds) - spam_n
        rows = "".join(
    f"""
    <tr>
        <td><span class='pill {"pspam" if p==1 else "pham"}'>
            {"SPAM" if p==1 else "HAM"}
        </span></td>
        <td style="font-family:DM Mono, monospace; font-size:11px; color:var(--mut);">{c}</td>
        <td style="font-size:12px; max-width:340px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;">{msg}</td>
    </tr>
    """
    for msg, p, c in zip(lines, preds, confs)
)
        display(HTML(f"""
        <div style="font-size:11px;color:var(--mut);margin-bottom:8px;">
          {len(lines)} messages &nbsp;&middot;&nbsp;
          <span style="color:#ff7a7a;">{spam_n} spam</span> &nbsp;&middot;&nbsp;
          <span style="color:#4ade80;">{ham_n} ham</span>
        </div>
        <table class="btable">
          <thead><tr><th>Result</th><th>Confidence</th><th>Message</th></tr></thead>
          <tbody>{rows}</tbody>
        </table>"""))

batch_btn.on_click(on_batch)
display(widgets.VBox([batch_box, batch_btn, batch_out], layout=widgets.Layout(gap='8px')))

# ═══════════════════ SECTION 3 — PERFORMANCE ════════════════════
display(HTML("<div class='sec-hd' style='margin-top:20px;'>// &#x1F4CA; Model Performance</div>"))

# Cards
winner_row = "".join(
    f"""<div class="pcard{' win' if n==best_name else ''}">
      <div class="ph">{n}{' <span class="wb">best</span>' if n==best_name else ''}</div>
      <div class="mr"><span class="mn">Accuracy</span> <span class="mv{' hi' if n==best_name else ''}">{r['accuracy']*100:.2f}%</span></div>
      <div class="mr"><span class="mn">Precision</span><span class="mv">{r['precision']*100:.2f}%</span></div>
      <div class="mr"><span class="mn">F1-Score</span> <span class="mv{' hi' if n==best_name else ''}">{r['f1']*100:.2f}%</span></div>
    </div>"""
    for n, r in results.items()
)
display(HTML(f"<div class='pgrid'>{winner_row}</div>"))

# Bar chart rows
COLORS = {'Logistic Regression':'#5b8fff', 'Naive Bayes':'#4ade80', 'KNN':'#b48cf5'}
def make_bars(metric_key, min_val, label):
    bars = "".join(
        f"""<div class="bgroup">
          <div class="bl"><span class="bn">{n}</span><span class="bv">{results[n][metric_key]*100:.2f}%</span></div>
          <div class="btrack"><div class="bfill" style="width:{int((results[n][metric_key]-min_val/100)/(1-min_val/100)*100)}%;background:{COLORS[n]};opacity:.85;"></div></div>
        </div>"""
        for n in results
    )
    return f"<div style='margin-bottom:14px;'><div style='font-size:10px;color:var(--mut);margin-bottom:7px;letter-spacing:.04em;'>{label}</div>{bars}</div>"

display(HTML(
    f"<div class='cbar-wrap'>"
    f"<div style='font-size:10px;color:var(--mut);text-transform:uppercase;letter-spacing:.06em;margin-bottom:13px;'>Metric Comparison</div>"
    + make_bars('accuracy',  88, 'Accuracy')
    + make_bars('precision', 88, 'Precision')
    + make_bars('f1',        55, 'F1-Score')
    + "</div>"
))

# Confusion matrix
cm = results[best_name]['cm']
display(HTML(f"""
<div class="cmw">
  <div style="font-size:10px;color:var(--mut);text-transform:uppercase;letter-spacing:.06em;margin-bottom:10px;">
    Confusion Matrix &mdash; {best_name} (best model)
  </div>
  <table class="cmt">
    <tr><th></th><th>Pred Ham</th><th>Pred Spam</th></tr>
    <tr><td style="font-size:10px;color:var(--mut);text-align:left;">Actual Ham</td>
        <td class="cok">{cm[0][0]}</td><td class="cerr">{cm[0][1]}</td></tr>
    <tr><td style="font-size:10px;color:var(--mut);text-align:left;">Actual Spam</td>
        <td class="cerr">{cm[1][0]}</td><td class="cok">{cm[1][1]}</td></tr>
  </table>
</div>"""))

display(HTML("<div class='sc-footer' style='margin-top:16px;'>&#x1F6E1; SMS Spam Classifier &middot; Osman Osama &middot; Egyptian Chinese University &middot; Data Mining #1</div>"))  

,Pred Ham,Pred Spam
Actual Ham,901,2
Actual Spam,20,111
